# ============================================================
# STEP 7 — DEEP LEARNING FULL CV (URL-Transformer + CNN+BiLSTM)
# ============================================================
# Unified script: runs BOTH architectures on BOTH tasks
# (binary + 5class) across ALL 5 domain-insulated folds.
# Produces per-fold results, CV averages, a comparison plot,
# and the final LaTeX comparison table.
#
# Runtime estimate (Colab T4):
#   URL-Transformer: ~18 min/fold × 5 × 2 tasks ≈ 3 h
#   CNN+BiLSTM:      ~10 min/fold × 5 × 2 tasks ≈ 1.7 h
#   Total: ~4-5 h  (run overnight or use range(1) for a quick check)
#
# Outputs:
#   step7_dl_results.csv          — per-fold metrics
#   step7_dl_comparison.png       — bar plot all models
#   step7c_comparison_table.tex   — LaTeX table for manuscript
# ============================================================

In [ ]:
import os, math, time, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score, matthews_corrcoef, roc_auc_score,
    precision_score, recall_score)

warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── PATHS ─────────────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/TUMC_dataset"
# dedup file has URL + fold + label_enc + class_enc
INPUT_FILE = os.path.join(DATA_DIR, "features_full_final_inductive_dedup.csv")
OUT_CSV    = os.path.join(DATA_DIR, "step7_dl_results.csv")
OUT_PLOT   = os.path.join(DATA_DIR, "step7_dl_comparison.png")
TEX_OUT    = os.path.join(DATA_DIR, "step7c_comparison_table.tex")
METRICS_CSV= os.path.join(DATA_DIR, "step3_all_metrics.csv")

# ── CONFIG ────────────────────────────────────────────────────
TASKS      = ["binary", "5class"]   # run both
ARCHS      = ["cnn_bilstm", "transformer"]  # run both
RUN_FOLDS  = list(range(5))         # full 5-fold CV
# To test quickly: RUN_FOLDS = [0]

MAX_LEN    = 200
EMBED_DIM  = 64
BATCH_SIZE = 512
MAX_EPOCHS = 15
PATIENCE   = 3
LR         = 1e-3

# ── VOCABULARY (shared by both architectures) ─────────────────
CHARS = ("abcdefghijklmnopqrstuvwxyz0123456789"
         ".-_/:?=&%@+~#!$*()[]{}|\\;,<>\"' çğıöşüâî")
char2idx = {c: i + 2 for i, c in enumerate(CHARS)}
PAD_IDX, UNK_IDX = 0, 1
VOCAB = len(char2idx) + 2

print("=" * 70)
print("STEP 7 — DEEP LEARNING FULL CV")
print(f"  Device: {DEVICE}  Folds: {RUN_FOLDS}")
print(f"  Architectures: {ARCHS}")
print(f"  Tasks: {TASKS}")
print("=" * 70)

# ════════════════════════════════════════════════════════════
# ENCODING + DATASET
# ════════════════════════════════════════════════════════════
def encode(url, ml=MAX_LEN):
    url = str(url).lower()[:ml]
    ids = [char2idx.get(c, UNK_IDX) for c in url]
    ids += [PAD_IDX] * (ml - len(ids))
    return ids

class URLDataset(Dataset):
    def __init__(self, urls, labels):
        self.X = np.array([encode(u) for u in urls], dtype=np.int64)
        self.y = np.array(labels, dtype=np.int64)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return torch.tensor(self.X[i]), torch.tensor(self.y[i])

# ════════════════════════════════════════════════════════════
# MODELS
# ════════════════════════════════════════════════════════════
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class URLTransformer(nn.Module):
    def __init__(self, vocab, n_classes, d=EMBED_DIM,
                 heads=4, layers=3, d_ff=128, drop=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab, d, padding_idx=PAD_IDX)
        self.pos   = PositionalEncoding(d)
        enc = nn.TransformerEncoderLayer(d, heads, d_ff, drop,
                                         activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, layers)
        self.dropout = nn.Dropout(drop)
        self.head = nn.Sequential(
            nn.Linear(d * 2, d), nn.GELU(), nn.Dropout(drop),
            nn.Linear(d, n_classes))
    def forward(self, x):
        mask = (x == PAD_IDX)
        h = self.pos(self.embed(x) * math.sqrt(EMBED_DIM))
        h = self.encoder(h, src_key_padding_mask=mask)
        hm = h.masked_fill(mask.unsqueeze(-1), 0).sum(1) / \
             (~mask).sum(1, keepdim=True).clamp(min=1)
        hx = h.masked_fill(mask.unsqueeze(-1), -1e9).max(1).values
        return self.head(self.dropout(torch.cat([hm, hx], 1)))

class CNNBiLSTM(nn.Module):
    def __init__(self, vocab, n_classes, embed=EMBED_DIM, hid=128):
        super().__init__()
        self.embed = nn.Embedding(vocab, embed, padding_idx=PAD_IDX)
        self.conv  = nn.Conv1d(embed, 128, 5, padding=2)
        self.relu  = nn.ReLU()
        self.lstm  = nn.LSTM(128, hid, batch_first=True, bidirectional=True)
        self.fc    = nn.Linear(hid * 2, n_classes)
    def forward(self, x):
        h = self.relu(self.conv(self.embed(x).permute(0, 2, 1))).permute(0, 2, 1)
        _, (hn, _) = self.lstm(h)
        return self.fc(torch.cat([hn[0], hn[1]], 1))

def build_model(arch, n_classes):
    if arch == "transformer": return URLTransformer(VOCAB, n_classes).to(DEVICE)
    return CNNBiLSTM(VOCAB, n_classes).to(DEVICE)

# ════════════════════════════════════════════════════════════
# TRAINING FUNCTION
# ════════════════════════════════════════════════════════════
def train_one_fold(arch, tr_urls, tr_y, te_urls, te_y, n_classes, cw):
    cw_t = torch.tensor(cw, dtype=torch.float32).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw_t)

    # 10% of train as internal validation for early stopping
    n_val = max(1, int(0.1 * len(tr_urls)))
    rng   = np.random.RandomState(SEED)
    perm  = rng.permutation(len(tr_urls))
    vi, ti = perm[:n_val], perm[n_val:]

    tr_dl  = DataLoader(URLDataset([tr_urls[i] for i in ti], tr_y[ti]),
                        batch_size=BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(URLDataset([tr_urls[i] for i in vi], tr_y[vi]),
                        batch_size=BATCH_SIZE)
    te_dl  = DataLoader(URLDataset(te_urls, te_y), batch_size=BATCH_SIZE)

    model = build_model(arch, n_classes)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    # Scheduler: OneCycleLR for CNN+BiLSTM, ReduceLROnPlateau for Transformer
    if arch == "cnn_bilstm":
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=LR, steps_per_epoch=len(tr_dl), epochs=MAX_EPOCHS)
        use_oclr = True
    else:
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode="min", factor=0.5, patience=1)
        use_oclr = False

    best_score, patience_cnt, best_state = -1., 0, None
    t0 = time.time()

    for ep in range(1, MAX_EPOCHS + 1):
        # ── train ──
        model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if use_oclr: sched.step()

        # ── validate ──
        model.eval()
        val_preds, val_loss = [], 0.
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                val_loss += crit(logits, yb).item()
                val_preds.extend(logits.argmax(1).cpu().tolist())
        val_loss /= max(len(val_dl), 1)
        if not use_oclr: sched.step(val_loss)

        val_f1 = f1_score(tr_y[vi], val_preds,
                          average="macro", zero_division=0)
        if val_f1 > best_score + 1e-4:
            best_score, patience_cnt = val_f1, 0
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"      early stop ep={ep} val_f1={best_score:.4f}")
                break

    model.load_state_dict(best_state)
    model.eval()
    preds, proba = [], []
    with torch.no_grad():
        for xb, _ in te_dl:
            logits = model(xb.to(DEVICE))
            preds.extend(logits.argmax(1).cpu().tolist())
            proba.extend(logits.softmax(1).cpu().numpy().tolist())

    elapsed = time.time() - t0
    return np.array(preds), np.array(proba), elapsed

# ════════════════════════════════════════════════════════════
# COMPUTE METRICS
# ════════════════════════════════════════════════════════════
def compute_metrics(task, y_true, y_pred, y_proba):
    m = {}
    m["f1_macro"]  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    m["mcc"]       = matthews_corrcoef(y_true, y_pred)
    m["precision"] = precision_score(y_true, y_pred, average="macro", zero_division=0)
    m["recall"]    = recall_score(y_true, y_pred, average="macro", zero_division=0)
    if task == "binary":
        m["auc"] = roc_auc_score(y_true, y_proba[:, 1])
    else:
        try:
            m["auc"] = roc_auc_score(y_true, y_proba,
                                      multi_class="ovr", average="macro")
        except Exception:
            m["auc"] = float("nan")
    return m

# ════════════════════════════════════════════════════════════
# LOAD DATA
# ════════════════════════════════════════════════════════════
print(f"\n[1] Loading {os.path.basename(INPUT_FILE)} ...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"  Rows: {len(df):,}  Folds: {sorted(df['fold'].unique())}")

enc_to_name = (df[["class_enc","class_final"]].drop_duplicates()
               .set_index("class_enc")["class_final"].to_dict())

# ════════════════════════════════════════════════════════════
# MAIN LOOP: arch × task × fold
# ════════════════════════════════════════════════════════════
all_rows = []
for arch in ARCHS:
    for task in TASKS:
        target_col = "label_enc" if task == "binary" else "class_enc"
        n_cls  = df[target_col].nunique()
        counts = df[target_col].value_counts().sort_index().values
        cw     = (df[target_col].shape[0] / (n_cls * counts)).astype(np.float32)

        print(f"\n{'='*60}")
        print(f"  {arch.upper()} | {task} | {n_cls} classes")
        print(f"  weights: {cw.round(2)}")
        print(f"{'='*60}")

        fold_metrics = []
        for fid in RUN_FOLDS:
            tr = df[df["fold"] != fid]
            te = df[df["fold"] == fid]
            print(f"\n  ── Fold {fid} (train={len(tr):,} test={len(te):,})")
            pred, proba, elapsed = train_one_fold(
                arch,
                tr["url"].tolist(), tr[target_col].values,
                te["url"].tolist(), te[target_col].values,
                n_cls, cw)
            m = compute_metrics(task, te[target_col].values, pred, proba)
            fold_metrics.append(m)
            key = "auc" if task == "binary" else "f1_macro"
            print(f"    {key}={m[key]:.4f}  f1_macro={m['f1_macro']:.4f}"
                  f"  MCC={m['mcc']:.4f}  ({elapsed:.0f}s)")

            # Save per-fold row
            all_rows.append({
                "arch": arch, "task": task, "fold": fid,
                **{k: round(v, 4) for k, v in m.items()},
                "elapsed_s": round(elapsed, 1)
            })

        # Summary
        keys = list(fold_metrics[0].keys())
        avg = {k: float(np.mean([fm[k] for fm in fold_metrics])) for k in keys}
        std = {k: float(np.std( [fm[k] for fm in fold_metrics])) for k in keys}
        print(f"\n  ── {arch} {task} CV SUMMARY ({len(RUN_FOLDS)} folds) ──")
        for k in ["f1_macro", "auc", "mcc"]:
            if k in avg:
                print(f"    {k}: {avg[k]:.4f} ± {std[k]:.4f}")

        # Save summary row
        all_rows.append({
            "arch": arch, "task": task, "fold": "mean",
            **{k: round(avg[k], 4) for k in keys},
            **{f"{k}_std": round(std[k], 4) for k in keys},
            "elapsed_s": round(np.sum([r["elapsed_s"] for r in all_rows
                                        if r["arch"]==arch and r["task"]==task
                                        and r["fold"]!=("mean")]), 1)
        })

        # Append to global leaderboard
        label = "Transformer" if arch == "transformer" else "CNN+BiLSTM"
        new_row = pd.DataFrame([{
            "model": label, "task": task, "variant": "raw_url_char",
            "fold": str(RUN_FOLDS), "n_folds": len(RUN_FOLDS),
            "f1_macro": round(avg["f1_macro"], 4),
            "mcc":      round(avg["mcc"], 4),
            "auc":      round(avg["auc"], 4),
        }])
        if os.path.exists(METRICS_CSV):
            lb = pd.read_csv(METRICS_CSV)
            lb = lb[~((lb["model"]==label) & (lb["task"]==task))]
            lb = pd.concat([lb, new_row], ignore_index=True)
        else:
            lb = new_row
        lb.to_csv(METRICS_CSV, index=False)

# Save all per-fold results
results_df = pd.DataFrame(all_rows)
results_df.to_csv(OUT_CSV, index=False)
print(f"\n[saved] {OUT_CSV}")

# ════════════════════════════════════════════════════════════
# COMPARISON PLOT
# ════════════════════════════════════════════════════════════
print("\n[Building comparison plot ...]")

# Results from Step 9 (classical ML, 5-fold CV)
CLASSICAL_5C = {
    "HistGB":    (0.9047, 0.0006),
    "XGBoost":   (0.8839, 0.0009),
    "LightGBM":  (0.8779, 0.0009),
    "RF":        (0.8619, 0.0007),
    "LogReg":    (0.2971, 0.0135),
}
CLASSICAL_BIN = {
    "HistGB":    (0.9999, 0.0006),
    "XGBoost":   (0.9999, 0.0008),
    "LightGBM":  (0.9999, 0.0007),
    "RF":        (1.0000, 0.0007),
    "LogReg":    (0.8936, 0.0135),
}

# Extract DL results from current run
mean_rows = results_df[results_df["fold"] == "mean"]
def get_dl(arch, task, metric):
    row = mean_rows[(mean_rows["arch"]==arch) & (mean_rows["task"]==task)]
    if len(row) == 0: return (float("nan"), float("nan"))
    std_col = f"{metric}_std" if f"{metric}_std" in row.columns else metric
    return (float(row[metric].iloc[0]),
            float(row[std_col].iloc[0]) if std_col != metric else 0.)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = {"Engineered (5-fold CV)": "#2196F3",
          "Deep Learning (5-fold CV)": "#FF5722"}

for ax, (task, cl_dict, metric, ylabel, title) in zip(axes, [
    ("5class", CLASSICAL_5C, "f1_macro", "Macro-F1 (5-class)", "5-Class Detection"),
    ("binary", CLASSICAL_BIN, "auc",     "AUC (binary)",       "Binary Detection"),
]):
    labels, vals, errs, cols = [], [], [], []
    # Classical ML
    for name, (v, s) in cl_dict.items():
        labels.append(name); vals.append(v); errs.append(s)
        cols.append("#1976D2" if name == "HistGB" else "#64B5F6")
    # Deep learning
    for arch, name in [("transformer", "Transformer"), ("cnn_bilstm", "CNN+BiLSTM")]:
        v, s = get_dl(arch, task, metric)
        labels.append(name); vals.append(v); errs.append(s)
        cols.append("#E64A19")

    x = np.arange(len(labels))
    bars = ax.bar(x, vals, color=cols, width=0.6,
                  yerr=errs, capsize=4, error_kw={"elinewidth":1.5})
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontweight="bold", fontsize=12)
    # value labels
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f"{v:.3f}", ha="center", va="bottom", fontsize=8)
    # separator line between classical and DL
    ax.axvline(4.5, color="gray", linestyle="--", linewidth=1, alpha=0.7)
    ax.text(2.2, ax.get_ylim()[0] + 0.01,
            "Engineered\nfeatures", ha="center", fontsize=8, color="#1565C0")
    ax.text(5.5, ax.get_ylim()[0] + 0.01,
            "Raw URL\nchars", ha="center", fontsize=8, color="#BF360C")
    lo = max(0, min(v for v in vals if not np.isnan(v)) - 0.05)
    ax.set_ylim(lo, min(1.01, max(v for v in vals if not np.isnan(v)) + 0.05))
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(axis="y", alpha=0.3)

# Legend
from matplotlib.patches import Patch
handles = [Patch(color="#1976D2", label="Best classical (HistGB)"),
           Patch(color="#64B5F6", label="Other classical ML (5-fold CV)"),
           Patch(color="#E64A19", label="Deep learning – raw URL chars (5-fold CV)")]
fig.legend(handles=handles, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.05), fontsize=9)
fig.suptitle("TUMC Model Comparison: Engineered Features vs Character-Level DL",
             fontweight="bold", fontsize=13)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.savefig(OUT_PLOT, dpi=160, bbox_inches="tight")
plt.close()
print(f"[saved] {OUT_PLOT}")

# ════════════════════════════════════════════════════════════
# LATEX COMPARISON TABLE
# ════════════════════════════════════════════════════════════
print("\n[Building LaTeX comparison table ...]")

# Pull final DL numbers
tr5_f1,  tr5_std  = get_dl("transformer",  "5class", "f1_macro")
tr5_mcc, _        = get_dl("transformer",  "5class", "mcc")
trb_auc, trb_std  = get_dl("transformer",  "binary", "auc")
trb_mcc, _        = get_dl("transformer",  "binary", "mcc")
cb5_f1,  cb5_std  = get_dl("cnn_bilstm",   "5class", "f1_macro")
cb5_mcc, _        = get_dl("cnn_bilstm",   "5class", "mcc")
cbb_auc, cbb_std  = get_dl("cnn_bilstm",   "binary", "auc")
cbb_mcc, _        = get_dl("cnn_bilstm",   "binary", "mcc")

n_folds_str = f"{len(RUN_FOLDS)}-fold CV" if len(RUN_FOLDS)==5 else f"Fold {RUN_FOLDS[0]} only"

CLASSICAL_FULL = [
    ("HistGB",             0.9047, 0.871, 0.9987, "5-fold CV", True),
    ("XGBoost",            0.8839, None,  0.9987, "5-fold CV", False),
    ("LightGBM",           0.8779, None,  0.9987, "5-fold CV", False),
    ("RandomForest",       0.8619, None,  1.0000, "5-fold CV", False),
    ("LogisticRegression", 0.2971, None,  0.8936, "5-fold CV", False),
]

lines = [
r"\begin{table}[t]", r"\centering",
(r"\caption{Full model comparison on TUMC. Classical ML: 5-fold domain-insulated CV on "
 r"135 engineered features. Character-level deep learning: raw URL string, "
 + n_folds_str + r". "
 r"Binary AUC and 5-class macro-$F_1$ are reported separately. "
 r"HistGB on engineered features dominates on the 5-class task; "
 r"all models converge near the binary-AUC ceiling, confirming that "
 r"the benign/malicious boundary is near-trivially separable.}"),
r"\label{tab:model-comparison-full}",
r"\footnotesize",
r"\begin{tabular}{lrrrr}",
r"\toprule",
r"Model & \multicolumn{2}{c}{5-class} & \multicolumn{2}{c}{Binary} \\",
r"\cmidrule(r){2-3}\cmidrule(l){4-5}",
r" & Macro-$F_1$ & MCC & AUC & Protocol \\",
r"\midrule",
r"\multicolumn{5}{l}{\textit{Engineered features (135 features after leaky exclusion)}} \\",
r"\midrule",
]
for name, f1, mcc, auc, proto, best in CLASSICAL_FULL:
    mcc_s = f"{mcc:.3f}" if mcc else "---"
    if best:
        lines.append(
            f"\\textbf{{{name}}} & \\textbf{{{f1:.3f}}} & "
            f"\\textbf{{{mcc_s}}} & \\textbf{{{auc:.4f}}} & {proto} \\\\")
    else:
        lines.append(
            f"{name} & {f1:.3f} & {mcc_s} & {auc:.4f} & {proto} \\\\")

lines += [
r"\midrule",
r"\multicolumn{5}{l}{\textit{Character-level deep learning (raw URL string, no engineered features)}} \\",
r"\midrule",
]

def fmt(v, std):
    if np.isnan(v): return "TBD"
    return f"{v:.3f}$\\pm${std:.3f}" if not np.isnan(std) else f"{v:.3f}"

lines.append(
    f"CNN+BiLSTM & {fmt(cb5_f1,cb5_std)} & {cb5_mcc:.3f} & "
    f"{fmt(cbb_auc,cbb_std)} & {n_folds_str} \\\\")
lines.append(
    f"URL-Transformer & {fmt(tr5_f1,tr5_std)} & {tr5_mcc:.3f} & "
    f"{fmt(trb_auc,trb_std)} & {n_folds_str} \\\\")

lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

with open(TEX_OUT, "w") as fh:
    fh.write("\n".join(lines))
print(f"[saved] {TEX_OUT}")

# ════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════
print(f"""
{'='*70}
STEP 7 COMPLETE — FULL CV RESULTS
{'='*70}

  5-CLASS COMPARISON (macro-F1):
    HistGB (engineered) : 0.905  [5-fold CV]
    CNN+BiLSTM (raw URL): {cb5_f1:.3f} ± {cb5_std:.3f}  [{n_folds_str}]
    Transformer (raw URL): {tr5_f1:.3f} ± {tr5_std:.3f}  [{n_folds_str}]
    Gap (trees - best DL): {0.9047 - max(cb5_f1, tr5_f1):.3f}

  BINARY COMPARISON (AUC):
    HistGB (engineered) : 0.9987  [5-fold CV]
    CNN+BiLSTM (raw URL): {cbb_auc:.4f} ± {cbb_std:.4f}  [{n_folds_str}]
    Transformer (raw URL): {trb_auc:.4f} ± {trb_std:.4f}  [{n_folds_str}]
    (all models converge near ceiling — binary task is near-trivially separable)

  THESIS INTERPRETATION:
    The {0.9047 - max(cb5_f1, tr5_f1):.2f} macro-F1 advantage of engineered features
    over the best character-level model quantifies the contribution
    of the hybrid feature pipeline. Graph-based campaign structure
    and Turkish linguistic features encode discriminative signal that
    raw URL characters alone cannot represent.

  Outputs saved:
    {OUT_CSV}
    {OUT_PLOT}
    {TEX_OUT}
{'='*70}
""")

STEP 7 — DEEP LEARNING FULL CV
  Device: cuda  Folds: [0, 1, 2, 3, 4]
  Architectures: ['cnn_bilstm', 'transformer']
  Tasks: ['binary', '5class']

[1] Loading features_full_final_inductive_dedup.csv ...
  Rows: 1,239,308  Folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

  CNN_BILSTM | binary | 2 classes
  weights: [3.25 0.59]

  ── Fold 0 (train=991,213 test=248,095)
      early stop ep=9 val_f1=0.9968
    auc=0.9999  f1_macro=0.9965  MCC=0.9931  (289s)

  ── Fold 1 (train=990,446 test=248,862)
    auc=0.9999  f1_macro=0.9967  MCC=0.9934  (482s)

  ── Fold 2 (train=994,741 test=244,567)
    auc=0.9999  f1_macro=0.9968  MCC=0.9936  (484s)

  ── Fold 3 (train=993,103 test=246,205)
    auc=0.9999  f1_macro=0.9977  MCC=0.9955  (484s)

  ── Fold 4 (train=987,729 test=251,579)
    auc=0.9999  f1_macro=0.9973  MCC=0.9947  (480s)

  ── cnn_bilstm binary CV SUMMARY (5 folds) ──
    f1_macro: 0.9970 ± 0.0005
    auc: 0.9999 ± 0.0000
    mcc: 0.9940 ± 0.0009

  CNN_BILST